# MorphSeq training-corpus census — first pass

This notebook distinguishes **gross acquisition opportunity**, **currently observed pipeline
products**, **confirmed post-QC samples**, and **projected post-QC samples**.

Primary units:

1. **Embryo-timepoint**: physical embryo × time, collapsed over z and channels.
2. **Embryo-plane observation**: physical embryo × time × z, channels collapsed.

Keyence/YX1 gross counts come from canonical frame inventories and `embryos_per_well`.
SeaHub gross counts are 8 embryos per included FOV. Projections are preliminary and should
always be labeled as projections in talk graphics.

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
from IPython.display import display

ROOT = Path.cwd()
DATA = ROOT / "corpus_census_data"
FIGURES = ROOT / "figures" / "corpus_census"
FIGURES.mkdir(parents=True, exist_ok=True)

wells = pd.read_csv(DATA / "well_census.csv")
datasets = pd.read_csv(DATA / "dataset_census.csv")
scope = pd.read_csv(DATA / "scope_summary.csv")
retention = pd.read_csv(DATA / "qc_retention_summary.csv")
conditions = pd.read_csv(DATA / "perturbation_conditions.csv")
atomic = pd.read_csv(DATA / "atomic_perturbations.csv")
sequencing = pd.read_csv(DATA / "sequencing_pairing_table.csv")

plt.rcParams.update({"figure.dpi": 120, "axes.spines.top": False, "axes.spines.right": False})
scope_order = ["Keyence", "YX1", "SeaHub"]
colors = {"Keyence": "#3B82F6", "YX1": "#F59E0B", "SeaHub": "#10B981"}

## Executive summary

In [ ]:
summary_cols = [
    "scope", "dataset_count", "gross_embryos", "gross_embryo_timepoints",
    "gross_embryo_z_observations", "z_expansion_factor",
    "confirmed_usable_timepoints", "projected_usable_timepoints",
    "projected_retention_rate",
]
display(scope[summary_cols].style.format({
    "gross_embryos": "{:,.0f}",
    "gross_embryo_timepoints": "{:,.0f}",
    "gross_embryo_z_observations": "{:,.0f}",
    "z_expansion_factor": "{:.1f}×",
    "confirmed_usable_timepoints": "{:,.0f}",
    "projected_usable_timepoints": "{:,.0f}",
    "projected_retention_rate": "{:.1%}",
}))
display(retention.style.format({
    "retention_rate": "{:.1%}",
    "retention_ci95_low": "{:.1%}",
    "retention_ci95_high": "{:.1%}",
}))

## Cumulative corpus growth by version

Each bar includes every acquisition dated on or before that version's cutoff. `v5` includes
the complete present-day **censusable** corpus.

- **Lower estimate:** strict current QC retention, extrapolated to every censusable dataset.
- **Upper estimate:** only death and structural mask-geometry exclusions.
- **Midpoint:** arithmetic midpoint of those bounds, used for the bar plots.

Both bounds assume that all censusable datasets eventually complete processing; they are not
counts of datasets that have already completed. SeaHub uses current detection completion as its
lower estimate and all reconciled embryos as its upper estimate because comparable downstream QC
is not yet available. The five-z comparison applies the fixed 5× expansion only to Keyence and
YX1 embryo-timepoints; SeaHub remains at one image per embryo. It is a projection, not measured
stack depth. Unique-perturbation trends count atomic perturbations from Keyence and YX1 only.

In [ ]:
from plot_training_corpus_versions import (
    REQUESTED_SLIDE_DIR,
    generate_version_figures,
)
from IPython.display import Image

# Plot controls.
COUNT_UNIT = "embryo_times"  # "embryos" or "embryo_times"
Z_SLICE_MULTIPLIER = 5

# The requested laptop path is used when the notebook runs there. Cluster execution falls
# back to a repository-local mirror because /Users/nick is not mounted on the cluster.
SLIDE_OUTPUT_DIR = (
    REQUESTED_SLIDE_DIR
    if REQUESTED_SLIDE_DIR.parent.is_dir()
    else ROOT / "data_census"
)

version_result = generate_version_figures(
    data_dir=DATA,
    output_dir=SLIDE_OUTPUT_DIR,
    count_unit=COUNT_UNIT,
    z_slice_multiplier=Z_SLICE_MULTIPLIER,
)
display(
    version_result["version_estimates"].style.format({
        "gross_count": "{:,.0f}",
        "lower_estimate": "{:,.0f}",
        "midpoint_estimate": "{:,.0f}",
        "upper_estimate": "{:,.0f}",
        "unique_perturbation_count": "{:,.0f}",
        "midpoint_5z_estimate": "{:,.0f}",
    })
)
display(
    version_result["projection_rates"].style.format({
        "lower_timepoint_rate": "{:.1%}",
        "upper_timepoint_rate": "{:.1%}",
        "lower_embryo_rate": "{:.1%}",
        "upper_embryo_rate": "{:.1%}",
    })
)
for figure_path in version_result["figure_paths"]:
    if figure_path.suffix.lower() == ".png":
        display(Image(filename=str(figure_path)))
print(f"Slide outputs: {version_result['output_dir']}")

## Corpus scale by acquisition scope

Solid bars show gross acquisition opportunity; hatched overlays show projected post-QC
availability. Confirmed counts are intentionally shown separately because pipeline completion
is still changing.

In [ ]:
plot = scope[scope.scope.isin(scope_order)].set_index("scope").reindex(scope_order)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for ax, gross_col, projected_col, title in [
    (axes[0], "gross_embryo_timepoints", "projected_usable_timepoints", "Embryo-timepoints"),
    (axes[1], "gross_embryo_z_observations", "projected_usable_z_observations", "Embryo × time × z"),
]:
    x = np.arange(len(plot))
    gross = plot[gross_col].to_numpy()
    projected = plot[projected_col].to_numpy()
    ax.bar(x, gross, color=[colors[s] for s in plot.index], alpha=.35, label="Gross")
    ax.bar(x, projected, color=[colors[s] for s in plot.index], label="Projected usable")
    ax.set_xticks(x, plot.index)
    ax.set_title(title)
    ax.set_ylabel("Observations")
    ax.ticklabel_format(axis="y", style="sci", scilimits=(0, 0))
axes[0].legend(frameon=False)
fig.suptitle("MorphSeq corpus size: gross versus projected post-QC")
fig.tight_layout()
fig.savefig(FIGURES / "corpus_size_by_scope.png", bbox_inches="tight")
fig.savefig(FIGURES / "corpus_size_by_scope.pdf", bbox_inches="tight")
plt.show()

## Processing and QC coverage

In [ ]:
coverage = (
    datasets.groupby("scope")
    .agg(
        datasets=("corpus_dataset_id", "nunique"),
        gross=("gross_embryo_timepoints", "sum"),
        observed=("observed_valid_snip_timepoints", "sum"),
        qc_evaluated=("qc_evaluated_timepoints", "sum"),
        confirmed_usable=("confirmed_usable_timepoints", "sum"),
    )
    .reindex(scope_order)
)
for col in ["observed", "qc_evaluated", "confirmed_usable"]:
    coverage[col + "_fraction"] = coverage[col] / coverage["gross"].replace(0, np.nan)
display(coverage.style.format("{:,.0f}", subset=["gross", "observed", "qc_evaluated", "confirmed_usable"])
        .format("{:.1%}", subset=["observed_fraction", "qc_evaluated_fraction", "confirmed_usable_fraction"]))

## Perturbation coverage

Conditions retain combinations (for example `environmental+chemical`). The atomic table
explodes combinations into unique temperature, chemical, and genetic targets.

In [ ]:
unique_atomic = (
    atomic.groupby(["scope", "perturbation_domain"])["atomic_perturbation"]
    .nunique()
    .unstack(fill_value=0)
    .reindex(scope_order)
)
display(unique_atomic)
ax = unique_atomic.plot(kind="bar", stacked=True, figsize=(8, 4),
                        color={"environmental": "#8B5CF6", "chemical": "#EF4444", "genetic": "#06B6D4"})
ax.set_ylabel("Unique atomic perturbations")
ax.set_xlabel("")
ax.set_title("Perturbation diversity by acquisition scope")
ax.legend(title="", frameon=False)
plt.tight_layout()
plt.savefig(FIGURES / "unique_perturbations_by_scope.png", bbox_inches="tight")
plt.savefig(FIGURES / "unique_perturbations_by_scope.pdf", bbox_inches="tight")
plt.show()

condition_volume = (
    wells.groupby(["scope", "perturbation_class"])["gross_embryo_timepoints"]
    .sum().unstack(fill_value=0).reindex(scope_order)
)
display(condition_volume.style.format("{:,.0f}"))

## Sequencing-linked imaging sets

In [ ]:
seq_display = sequencing.loc[
    sequencing["pairing_status"].ne("not_linked"),
    [
        "scope", "corpus_dataset_id", "pairing_status",
        "linked_well_or_embryo_count", "total_well_or_embryo_count",
        "collection_name", "sequencing_modality", "sequencing_dataset_id",
        "pairing_granularity", "evidence_source", "curator_notes",
    ],
]
display(seq_display)
print("Edit sequencing_pairing_overrides.csv and refresh to curate Keyence/YX1 links.")

## Classification review queue

In [ ]:
review = (
    wells.loc[wells["classification_needs_review"].astype(bool),
              ["scope", "corpus_dataset_id", "perturbation", "genotype",
               "chem_perturbation", "temperature", "perturbation_class",
               "condition_signature", "classification_review_reason"]]
    .drop_duplicates()
    .sort_values(["scope", "corpus_dataset_id"])
)
print(f"{len(review):,} unique labels/conditions need review")
display(review.head(200))

## Detailed dataset census

In [ ]:
display(
    datasets.sort_values(["scope", "gross_embryo_timepoints"], ascending=[True, False])
    .style.format({
        "gross_embryo_timepoints": "{:,.0f}",
        "gross_embryo_z_observations": "{:,.0f}",
        "z_expansion_factor": "{:.1f}×",
        "qc_coverage_fraction": "{:.1%}",
        "confirmed_retention_rate": "{:.1%}",
        "projected_usable_timepoints": "{:,.0f}",
    })
)

## Important limitations

- Gross Keyence/YX1 counts use metadata `embryos_per_well`; where a frame inventory is
  missing, within-dataset median time/z depth is used and flagged.
- Where `embryos_per_well` is absent or too small, the peak number of concurrently detected
  embryos in that well supplies a conservative lower-bound embryo count. This avoids
  undercounting multi-embryo YX1 wells but can inherit detection errors.
- Projection rates describe the datasets that have completed QC; they may not represent
  the hardest unfinished datasets.
- SeaHub currently has detection-completion retention only, not completed downstream QC.
- “Sequencing linked” indicates a collection/condition link, not necessarily the identical
  individual embryo.
- Perturbation classes are rule-based. Review the generated classification queue before
  using exact diversity counts in a talk.

## Image × sequencing pairing

Only experiments that are part of the paired image+sequencing program appear here. Every other
Keyence experiment is excluded outright — 60 of 97 — because they will never have sequencing.
Their images are not "outstanding"; they are simply not part of this program.

**Keyence** (37 experiments, enumerated by hand):

| Keyence datasets | status | sequencing project |
|---|---|---|
| `20260319`, `20260320`, `20260324`, `20260331`, `20260414`, `20260415`, `20260416` (cilia crispant, `cep290`, `b9d2`) — 22 datasets | sequenced | `GENE14` |
| all six `20250612_*` ER-stress plates | sequenced | `GENE7` |
| `20240813_24hpf/_30hpf/_36hpf` (`_extras` excluded) | sequenced | — |
| all six `20260702_hotchem` plates | **outstanding** | — |
| `20260724` | **outstanding**, and absent from the census | — |

**SeaHub** (all 20 experiments) pairs by name against the pseudobulk inventory under
`PSEUDOBULK_DIR`: a project is sequenced when `<PROJECT>.pseudobulk.npz` exists, otherwise
outstanding.

Validation flags are used *only* to decide which images are valid, never to decide whether an
image is paired.

In [ ]:
import ast
import re
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from IPython.display import display

from plot_training_corpus_versions import PERTURBATION_COLORS

ROOT = Path.cwd()
DATA = ROOT / "corpus_census_data"
SEQ_FIGURES = ROOT / "figures" / "sequencing_pairing"
SEQ_FIGURES.mkdir(parents=True, exist_ok=True)

PSEUDOBULK_DIR = Path("/net/trapnell/vol1/cliff/data/2026_cds_training_data/pseudobulk/"
                      "v3.1.1/nobackup/results/pseudobulk")

PAIR_SCOPES = ["Keyence", "SeaHub"]
HATCH, HATCH_ALPHA = "///", 0.32

# The only Keyence experiments that are part of the paired program. Prefix match.
KEYENCE_SEQUENCED = {
    "GENE14": ("20260319", "20260320", "20260324", "20260331",
               "20260414", "20260415", "20260416"),
    "GENE7": ("20250612_",),
    "20240813*": ("20240813_24hpf", "20240813_30hpf", "20240813_36hpf"),
}
KEYENCE_OUTSTANDING = {"20260702*": ("20260702",)}
NOT_YET_CENSUSED = ["20260724"]   # outstanding, and no census rows to count


def project_of(filename):
    """GENE10.pseudobulk.npz -> GENE10; CHEM11_24.pseudobulk.npz -> CHEM11."""
    return re.sub(r"_\d+$", "", re.sub(r"\.pseudobulk\.npz$", "", filename))


def parse_atomic(x):
    try:
        v = ast.literal_eval(x)
        return v if isinstance(v, list) else []
    except Exception:
        return []


def expand(spec, universe):
    return {label: sorted(i for i in universe if i.startswith(pre))
            for label, pre in spec.items()}


sequenced_projects = sorted({project_of(p.name) for p in PSEUDOBULK_DIR.iterdir()
                             if p.name.endswith(".pseudobulk.npz")})

pair_wells = pd.read_csv(DATA / "well_census.csv", low_memory=False)
pair_datasets = pd.read_csv(DATA / "dataset_census.csv")
for _df in (pair_wells, pair_datasets):
    _df["corpus_dataset_id"] = _df["corpus_dataset_id"].astype(str)

_kid = set(pair_datasets.loc[pair_datasets["scope"].eq("Keyence"), "corpus_dataset_id"])
_sid = set(pair_datasets.loc[pair_datasets["scope"].eq("SeaHub"), "corpus_dataset_id"])
keyence_sequenced_sets = expand(KEYENCE_SEQUENCED, _kid)
keyence_outstanding_sets = expand(KEYENCE_OUTSTANDING, _kid)
keyence_sequenced = {i for ids in keyence_sequenced_sets.values() for i in ids}
keyence_outstanding = {i for ids in keyence_outstanding_sets.values() for i in ids}
seahub_sequenced = {i for i in _sid if i in sequenced_projects}

in_program = keyence_sequenced | keyence_outstanding | _sid
sequenced_ids = keyence_sequenced | seahub_sequenced

# Restrict to the paired program, then flag pairing.
pair_wells = pair_wells[pair_wells["corpus_dataset_id"].isin(in_program)].copy()
pair_datasets = pair_datasets[pair_datasets["corpus_dataset_id"].isin(in_program)].copy()
for _df in (pair_wells, pair_datasets):
    _df["paired"] = _df["corpus_dataset_id"].isin(sequenced_ids)

# Validated image = reconciled embryo. These Keyence plates are single-timepoint, so their
# validated snip count equals their reconciled embryo count -- one definition covers both scopes.
VALID_COL = "observed_unique_physical_embryos"

print("Keyence — in the paired program:")
for label, ids in keyence_sequenced_sets.items():
    print(f"  sequenced    {label:<12} {len(ids):>2} datasets")
for label, ids in keyence_outstanding_sets.items():
    print(f"  OUTSTANDING  {label:<12} {len(ids):>2} datasets")
print(f"  OUTSTANDING  {', '.join(NOT_YET_CENSUSED):<12}  not in census, not counted")
print(f"  excluded from the figures entirely: {len(_kid - in_program)} Keyence experiments")
print()
print("SeaHub — pseudobulk inventory:")
print(f"  sequenced   ({len(seahub_sequenced):>2}): " + ", ".join(sorted(seahub_sequenced)))
print(f"  OUTSTANDING ({len(_sid - seahub_sequenced):>2}): "
      + ", ".join(sorted(_sid - seahub_sequenced)))
print()

# ---- validated images by scope x perturbation_class x pairing
images_by_class = (
    pair_wells.pivot_table(index=["scope", "perturbation_class"], columns="paired",
                           values=VALID_COL, aggfunc="sum", fill_value=0)
    .rename(columns={True: "sequenced", False: "imaging_only"})
)

# ---- unique atomic perturbations by scope x domain x pairing
_p = (
    pair_wells.assign(ap=pair_wells["atomic_perturbations"].map(parse_atomic))
    [["scope", "paired", "ap"]]
    .explode("ap")
    .dropna(subset=["ap"])
)
_p["domain"] = _p["ap"].map(lambda t: t[0])
_p["name"] = _p["ap"].map(lambda t: str(t[1]))
perts_by_domain = (
    _p.groupby(["scope", "domain", "name"])["paired"].any()
    .reset_index()
    .groupby(["scope", "domain"])["paired"]
    .agg(sequenced="sum", total="size")
    .assign(imaging_only=lambda d: d["total"] - d["sequenced"])
)

for _tbl in (images_by_class, perts_by_domain):
    for _c in ("sequenced", "imaging_only"):
        if _c not in _tbl:
            _tbl[_c] = 0.0

display(images_by_class[["sequenced", "imaging_only"]].style.format("{:,.0f}"))
display(perts_by_domain[["sequenced", "imaging_only"]])

for _name, _tbl in (("validated images", images_by_class),
                    ("atomic perturbations", perts_by_domain)):
    _g = _tbl.groupby(level="scope")[["sequenced", "imaging_only"]].sum()
    print(f"{_name}:")
    for _s in PAIR_SCOPES:
        _seq, _img = _g.loc[_s, "sequenced"], _g.loc[_s, "imaging_only"]
        print(f"  {_s:<8} {_seq + _img:>7,.0f} total  {_seq:>7,.0f} sequenced"
              f"  ({_seq / (_seq + _img):.0%})")
print()
_gross = pair_datasets.pivot_table(index="scope", columns="paired", values="gross_embryos",
                                   aggfunc="sum", fill_value=0)
print("gross embryos, for reference (validated counts above exclude unprocessed embryos):")
for _s in PAIR_SCOPES:
    print(f"  {_s:<8} {_gross.loc[_s].sum():>7,.0f} gross  "
          f"{_gross.loc[_s, True]:>7,.0f} sequenced")

In [ ]:
def paired_stacked_bar(table, order, ylabel, outstem, min_label_frac=0.035, legend_cols=3):
    """Two stacked bars (Keyence, SeaHub); hatched portion of each block = sequencing outstanding."""
    present = [c for c in order if c in {lvl for _, lvl in table.index}]
    totals = table.groupby(level="scope")[["sequenced", "imaging_only"]].sum().sum(axis=1)
    span = totals.max()

    fig, ax = plt.subplots(figsize=(8.2, 6.6))
    xs = np.arange(len(PAIR_SCOPES))
    for xi, scope_name in zip(xs, PAIR_SCOPES):
        bottom = 0.0
        for cat in present:
            if (scope_name, cat) not in table.index:
                continue
            row = table.loc[(scope_name, cat)]
            color = PERTURBATION_COLORS.get(cat, "#374151")
            for value, is_hatch in ((row["sequenced"], False), (row["imaging_only"], True)):
                if value <= 0:
                    continue
                ax.bar(xi, value, bottom=bottom, width=0.58, color=color,
                       alpha=HATCH_ALPHA if is_hatch else 1.0,
                       hatch=HATCH if is_hatch else None,
                       edgecolor=color if is_hatch else "white", linewidth=0.9)
                if value >= span * min_label_frac:
                    ax.text(xi, bottom + value / 2, f"{value:,.0f}", ha="center", va="center",
                            fontsize=12.75, fontweight="bold",
                            color="#1F2937" if is_hatch else "white")
                bottom += value
        seq = table.loc[scope_name, "sequenced"].sum()
        ax.text(xi, bottom + span * 0.02,
                f"{bottom:,.0f}\n{seq / bottom:.0%} sequenced" if bottom else "0",
                ha="center", va="bottom", fontsize=14.25, fontweight="bold", color="#334155")

    n_seq = {s: int(pair_datasets.loc[pair_datasets["scope"].eq(s), "paired"].sum())
             for s in PAIR_SCOPES}
    n_tot = {s: int(pair_datasets["scope"].eq(s).sum()) for s in PAIR_SCOPES}
    ax.set_xticks(xs, [f"{s}\n{n_seq[s]} of {n_tot[s]} sequenced"
                       for s in PAIR_SCOPES], fontsize=15)
    ax.set_ylabel(ylabel, fontsize=20)
    ax.tick_params(axis="y", labelsize=15)
    ax.set_ylim(0, span * 1.16)
    ax.set_xlim(-0.62, len(PAIR_SCOPES) - 0.38)
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(axis="x", length=0)

    handles = [Patch(facecolor=PERTURBATION_COLORS.get(c, "#374151"), edgecolor="white", label=c)
               for c in present]
    handles.append(Patch(facecolor="#94A3B8", alpha=HATCH_ALPHA, hatch=HATCH,
                         edgecolor="#64748B", label="imaging only (sequencing outstanding)"))
    ax.legend(handles=handles, frameon=False, fontsize=12.75, loc="upper center",
              bbox_to_anchor=(0.5, -0.17), ncol=legend_cols, handlelength=1.6,
              columnspacing=1.6, labelspacing=0.7)

    fig.savefig(SEQ_FIGURES / f"{outstem}.png", dpi=200, bbox_inches="tight")
    fig.savefig(SEQ_FIGURES / f"{outstem}.pdf", bbox_inches="tight")
    plt.show()


CLASS_ORDER = list(PERTURBATION_COLORS)      # control, environmental, chemical, genetic, ...
DOMAIN_ORDER = ["environmental", "chemical", "genetic"]

paired_stacked_bar(
    perts_by_domain, DOMAIN_ORDER,
    "unique atomic perturbations",
    "paired_diversity_atomic_perturbations",
    legend_cols=2,
)
print("wrote paired_diversity_atomic_perturbations.{png,pdf}")

In [ ]:
paired_stacked_bar(
    images_by_class, CLASS_ORDER,
    "validated images",
    "paired_validated_images_by_perturbation_class",
    legend_cols=4,
)
images_by_class.to_csv(SEQ_FIGURES / "validated_images_pairing.csv")
perts_by_domain.to_csv(SEQ_FIGURES / "atomic_perturbation_pairing.csv")
print("wrote paired_validated_images_by_perturbation_class.{png,pdf} + both tables")

### Caveats

- **`20240813*` is counted as sequenced on curation alone.** No pseudobulk file name identifies
  it. `20250612*` → `GENE7` and the cilia series → `GENE14` are both corroborated by a file of
  that name existing.
- **`GENE7` covers imaging in both scopes.** SeaHub `GENE7` is `pax1a`/`pax1b`/`pax9`; the
  Keyence `20250612` plates are `atf6`/`ctcf`/`wfs1a-wfs1b`. One sequencing project, two scopes.
- **`20260724` is outstanding and has no census rows**, so it contributes nothing to either bar.
  The Keyence outstanding side is therefore understated by that experiment's embryos.
- **Validated counts exclude unprocessed embryos.** Keyence has 3,710 gross embryos in the
  program but only 2,361 reconciled, because 11 of the 22 cilia datasets have not been through
  detection/QC yet. SeaHub: 8,152 gross, 8,064 reconciled.
- Keyence chemical perturbations show as 0 of 2 sequenced because `hsp90` and `mTor` appear only
  in the `20260702_hotchem` plates, which are outstanding.
- Sequencing existence is inferred from a pseudobulk file being present, not from library QC.
  Nothing inside those files was read.
- Five pseudobulk projects (`CHEM1.1`, `CHEM9`–`CHEM12`) and `reference` have no corpus imaging
  attributed to them.